# Gold Medal Tabular Modeling Baseline

競馬予測への金メダル解法戦略の適用。

16コンペ（Optiver, AMEX, IEEE Fraud, Home Credit, Ubiquant, Child Mind, ICR, Equity HCT, Enefit, Mitsui 等）の
金メダル解法から抽出した共通パターンを keiba-vpn に実装する。

## 実装する戦略

| 戦略 | 参考コンペ |
|---|---|
| LGBM + XGB + CatBoost 3モデルアンサンブル | 全コンペ共通 |
| ラグ特徴・ローリング統計 | Optiver, Enefit, Ubiquant |
| グループ集約（同日同開催コンテキスト） | Ubiquant time_id 集約 |
| 2段階分解（分類 + ランキング） | Equity HCT |
| Purged GroupKFold（レース日付でグループ化） | IEEE Fraud, Jane Street |
| 多シード平均（ノイズ安定化） | Child Mind, ICR |
| Adversarial Validation（分布シフト検出） | IEEE Fraud, AMEX |
| 予測後処理（レース内ランク整合） | Optiver zero-sum, IEEE Fraud UID averaging |


## 0. Setup

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)

# プロジェクトルート解決
_root = Path.cwd().resolve()
for _ in range(16):
    if (_root / "requirements.txt").is_file() and (_root / ".env.example").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("keiba-vpn ルートが見つかりません")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

REPO_ROOT = _root
print("REPO_ROOT:", REPO_ROOT)

In [ ]:
# ライブラリ確認
import importlib

REQUIRED = ["lightgbm", "xgboost", "catboost", "sklearn", "optuna"]
OPTIONAL = ["shap"]

for lib in REQUIRED:
    try:
        m = importlib.import_module(lib)
        print(f"  OK  {lib} {getattr(m, '__version__', '')}")
    except ImportError:
        print(f"  NG  {lib} — pip install {lib}")

for lib in OPTIONAL:
    try:
        m = importlib.import_module(lib)
        print(f"  OK  {lib} {getattr(m, '__version__', '')} (optional)")
    except ImportError:
        print(f"  --  {lib} not installed (optional — SHAP 解釈は skip)")

## 1. データ読み込み

既存パイプライン `build_layer_a_dataframe()` を使って Layer-A 学習母表を構築する。  
事前に `python -m pipeline.build_layer_a_dataset` を実行していれば parquet キャッシュを利用可。

In [ ]:
LAYER_A_CACHE = REPO_ROOT / "data" / "local" / "modeling" / "layer_a_train.parquet"

if LAYER_A_CACHE.exists():
    print("キャッシュから読み込み:", LAYER_A_CACHE)
    df_raw = pd.read_parquet(LAYER_A_CACHE)
else:
    print("キャッシュなし → build_layer_a_dataframe() で構築")
    from src.pipeline.models.layer_a_dataset import build_layer_a_dataframe
    df_raw = build_layer_a_dataframe(base_dir=REPO_ROOT)

print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns[:20])} ...")
df_raw.head(3)

In [ ]:
# 日付列の確認と型変換
DATE_COLS = [c for c in ["race_date", "kaisai_date", "date"] if c in df_raw.columns]
DATE_COL = DATE_COLS[0] if DATE_COLS else None
print("日付列:", DATE_COL)

if DATE_COL:
    df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL], errors="coerce")
    print(f"期間: {df_raw[DATE_COL].min()} 〜 {df_raw[DATE_COL].max()}")
    print(f"レース数: {df_raw['race_id'].nunique():,}")
    print(f"馬数（延べ）: {len(df_raw):,}")

# ターゲット確認
TARGET_COLS = [c for c in ["finish_position", "rank", "finish_pos"] if c in df_raw.columns]
TARGET_COL = TARGET_COLS[0] if TARGET_COLS else None
print("\nターゲット列:", TARGET_COL)
if TARGET_COL:
    print(df_raw[TARGET_COL].value_counts().sort_index().head(20))

## 2. 特徴量エンジニアリング（金メダル戦略）

### 2-1. ラグ特徴・ローリング統計

> Optiver 1位: "Feature engineering was the primary driver of performance."  
> Enefit: 2〜14日前のラグが全解法で共通  
> Home Credit: "time-sliced windows rather than static aggregates"

In [ ]:
def add_horse_sequential_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """
    馬単位の時系列特徴量を追加する。
    - 前走タイム・着順ラグ (lag 1〜3)
    - 直近 3/5 走ローリング統計（平均・標準偏差）
    - 前走からの日数差
    
    ※ leakage 防止: sort → shift のみ使用（当日情報は含めない）
    """
    df = df.copy()
    df = df.sort_values(["horse_id", date_col]).reset_index(drop=True)

    NUMERIC_COLS = [c for c in [
        "finish_time_sec", "last_3f_sec", "speed_max", "speed_avg",
        "weight_carried", "horse_weight", "speed_recent",
    ] if c in df.columns]

    target_like = TARGET_COL  # 'finish_position' 等

    grp = df.groupby("horse_id", sort=False)

    for col in NUMERIC_COLS + ([target_like] if target_like else []):
        for lag in [1, 2, 3]:
            df[f"{col}_lag{lag}"] = grp[col].shift(lag)

    for col in NUMERIC_COLS + ([target_like] if target_like else []):
        for window in [3, 5]:
            rolled = grp[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
            df[f"{col}_roll{window}mean"] = rolled
            rolled_std = grp[col].transform(lambda s: s.shift(1).rolling(window, min_periods=2).std())
            df[f"{col}_roll{window}std"] = rolled_std

    if date_col in df.columns:
        days_since = grp[date_col].transform(lambda s: (s - s.shift(1)).dt.days)
        df["days_since_prev_race"] = days_since

    return df


print("add_horse_sequential_features() 定義完了")

### 2-2. グループ集約特徴（同日同開催コンテキスト）

> Ubiquant 1位: "time_id aggregation — CV 0.141 → 0.154"  
> IEEE Fraud: "frequency encoding on high-cardinality features"

競馬では `race_date × venue_code` が Ubiquant の `time_id` に相当。  
同じ馬場・同じ日の他レースのオッズや速度指数は強い文脈特徴になる。

In [ ]:
def add_race_context_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """
    同日同開催（race_date × venue_code）内の集約特徴を追加する。
    Ubiquant の time_id 集約に相当。
    """
    df = df.copy()

    venue_col = next((c for c in ["venue_code", "course_code", "venue"] if c in df.columns), None)
    group_key = [date_col, venue_col] if venue_col else [date_col]

    AGGREGATE_COLS = [c for c in [
        "speed_max", "speed_avg", "weight_carried",
    ] if c in df.columns]

    for col in AGGREGATE_COLS:
        ctx_mean = df.groupby(group_key)[col].transform("mean")
        ctx_std = df.groupby(group_key)[col].transform("std")
        df[f"{col}_ctx_mean"] = ctx_mean
        df[f"{col}_ctx_std"] = ctx_std
        # 同コンテキスト内での偏差
        df[f"{col}_ctx_dev"] = df[col] - ctx_mean

    # レース内出走頭数
    df["n_horses_in_race"] = df.groupby("race_id")["race_id"].transform("count")

    # 騎手・調教師の頻度エンコーディング（high cardinality 対策）
    for entity_col in ["jockey_id", "trainer_id"]:
        if entity_col in df.columns:
            freq = df[entity_col].map(df[entity_col].value_counts())
            df[f"{entity_col}_freq"] = freq

    return df


print("add_race_context_features() 定義完了")

### 2-3. ドメイン特有の特徴量

> Home Credit: "Domain-Specific Metrics — Interest Rate, Credit Utilization"  
> Enefit: `installed_capacity * solar_radiation / (temperature + 273.15)`

In [ ]:
def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    競馬ドメイン固有の特徴量を追加する。
    - 斤量負担率（weight_carried / horse_weight）
    - 前走着差の回復指標
    - 直近ラグ間のトレンド（速度指数の傾き）
    """
    df = df.copy()

    # 斤量負担率（軽い馬ほど斤量の影響が大きい）
    if "weight_carried" in df.columns and "horse_weight" in df.columns:
        hw = pd.to_numeric(df["horse_weight"], errors="coerce")
        wc = pd.to_numeric(df["weight_carried"], errors="coerce")
        df["burden_ratio"] = wc / hw.replace(0, np.nan)

    # 速度指数トレンド（lag1 - lag2: 上昇中か下降中か）
    if "speed_max_lag1" in df.columns and "speed_max_lag2" in df.columns:
        df["speed_trend_1v2"] = df["speed_max_lag1"] - df["speed_max_lag2"]
    if "speed_max_lag2" in df.columns and "speed_max_lag3" in df.columns:
        df["speed_trend_2v3"] = df["speed_max_lag2"] - df["speed_max_lag3"]

    # 着順ラグ間のトレンド
    if TARGET_COL and f"{TARGET_COL}_lag1" in df.columns and f"{TARGET_COL}_lag2" in df.columns:
        df["rank_trend_1v2"] = df[f"{TARGET_COL}_lag2"] - df[f"{TARGET_COL}_lag1"]  # 正 = 改善

    return df


print("add_domain_features() 定義完了")

In [ ]:
# 特徴量パイプライン実行
if DATE_COL and "horse_id" in df_raw.columns:
    df_fe = add_horse_sequential_features(df_raw, DATE_COL)
    df_fe = add_race_context_features(df_fe, DATE_COL)
    df_fe = add_domain_features(df_fe)
    print(f"特徴量追加後 shape: {df_fe.shape}")
    new_cols = [c for c in df_fe.columns if c not in df_raw.columns]
    print(f"新規特徴量数: {len(new_cols)}")
    print("サンプル:", new_cols[:15])
else:
    df_fe = df_raw.copy()
    print("WARNING: horse_id または date_col が見つからないため特徴量追加をスキップ")

## 3. 学習データ準備

### 3-1. ターゲット定義（2段階分解）

> Equity HCT: "分類（事象有無）+ 回帰（時刻）を別モデルで学習し結合" → 全金メダル解法が採用

競馬への適用:
- **Stage A（分類）**: 3着以内に入るか（0/1）
- **Stage B（ランキング）**: 着順のランク学習（LambdaRank）

In [ ]:
# ターゲットが存在する行のみ使用
if TARGET_COL is None:
    raise ValueError(f"ターゲット列が見つかりません。df_fe.columns: {list(df_fe.columns)[:30]}")

df_train = df_fe[df_fe[TARGET_COL].notna()].copy()
df_train[TARGET_COL] = pd.to_numeric(df_train[TARGET_COL], errors="coerce")
df_train = df_train[df_train[TARGET_COL].notna()]

# Stage A: 3着以内バイナリターゲット
PLACE_THRESHOLD = 3
df_train["target_place"] = (df_train[TARGET_COL] <= PLACE_THRESHOLD).astype(int)

# Stage B: ランクターゲット（着順そのもの）
df_train["target_rank"] = df_train[TARGET_COL].astype(float)

print(f"学習データ: {df_train.shape}")
print(f"\n3着以内率: {df_train['target_place'].mean():.3f}")
print(f"着順分布:")
print(df_train[TARGET_COL].value_counts().sort_index().head(20))

In [ ]:
# 特徴量列の選定
EXCLUDE_COLS = {
    TARGET_COL, "target_place", "target_rank",
    "race_id", "horse_id", "horse_number",
    "horse_name", "jockey_name", "trainer_name",
    # leakage になる列
    "win_odds", "popularity", "finish_time_sec", "last_3f_sec",
    "finish_position", "rank", "finish_pos",
}
if DATE_COL:
    EXCLUDE_COLS.add(DATE_COL)

FEATURE_COLS = [
    c for c in df_train.columns
    if c not in EXCLUDE_COLS
    and not c.startswith("target_")
]

print(f"特徴量数: {len(FEATURE_COLS)}")
print("先頭20列:", FEATURE_COLS[:20])

X = df_train[FEATURE_COLS]
y_place = df_train["target_place"]
y_rank = df_train["target_rank"]

print(f"\nX shape: {X.shape}")
print(f"欠損率 > 50% の列数: {(X.isnull().mean() > 0.5).sum()}")

## 4. Purged GroupKFold（時系列バリデーション）

> IEEE Fraud: "train on months 1-4, skip month 5, predict month 6 — to mimic train/test gap"  
> Jane Street: "Purged Group Time-Series Split"  
> 共通原則: **「CV を信じろ。PublicLB は信じるな」**

In [ ]:
from sklearn.model_selection import GroupKFold


class PurgedGroupTimeSeriesSplit:
    """
    時系列データ用 Purged GroupKFold。
    
    - groups: race_date をエンコードした整数（古い日付 = 小さい値）
    - purge_gap: 訓練最終日〜検証開始日のギャップ（日数）。情報リーク防止。
    
    参考: IEEE Fraud 1位「45-day gap」、Optiver「5-day gap」
    """

    def __init__(self, n_splits: int = 5, purge_gap_days: int = 7):
        self.n_splits = n_splits
        self.purge_gap_days = purge_gap_days

    def split(self, X, y=None, groups=None):
        """
        groups: pd.Series of datetime (race_date)
        """
        if groups is None:
            raise ValueError("groups (race_date) が必要です")

        groups = pd.to_datetime(groups)
        unique_dates = sorted(groups.unique())
        n_dates = len(unique_dates)
        fold_size = n_dates // (self.n_splits + 1)

        for fold in range(self.n_splits):
            train_end_idx = fold_size * (fold + 1)
            val_start_idx = train_end_idx + 1  # 最低1日ギャップ
            val_end_idx = min(train_end_idx + fold_size, n_dates)

            if val_start_idx >= n_dates:
                break

            train_end_date = unique_dates[train_end_idx - 1]
            val_start_date = unique_dates[val_start_idx]

            # purge_gap_days 以上離れているか確認
            actual_gap = (val_start_date - train_end_date).days
            if actual_gap < self.purge_gap_days:
                # ギャップが不足 → 検証開始をずらす
                from pandas import Timedelta
                target_val_start = train_end_date + Timedelta(days=self.purge_gap_days)
                filtered = [d for d in unique_dates if d >= target_val_start]
                if not filtered:
                    continue
                val_start_date = filtered[0]

            val_end_date = unique_dates[min(val_end_idx - 1, n_dates - 1)]

            train_mask = groups <= train_end_date
            val_mask = (groups >= val_start_date) & (groups <= val_end_date)

            train_idx = np.where(train_mask)[0]
            val_idx = np.where(val_mask)[0]

            if len(train_idx) == 0 or len(val_idx) == 0:
                continue

            yield train_idx, val_idx


# フォールド確認
if DATE_COL:
    splitter = PurgedGroupTimeSeriesSplit(n_splits=5, purge_gap_days=7)
    folds = list(splitter.split(X, groups=df_train[DATE_COL]))
    print(f"有効フォールド数: {len(folds)}")
    for i, (tr, va) in enumerate(folds):
        tr_dates = df_train[DATE_COL].iloc[tr]
        va_dates = df_train[DATE_COL].iloc[va]
        gap = (va_dates.min() - tr_dates.max()).days
        print(f"  Fold {i+1}: train={len(tr):,}件 ({tr_dates.min().date()}〜{tr_dates.max().date()}) "
              f"| gap={gap}日 | val={len(va):,}件 ({va_dates.min().date()}〜{va_dates.max().date()})")

## 5. Adversarial Validation（分布シフト検出）

> IEEE Fraud 1位・AMEX 11位: "Adversarial Validation — drop features with high train/test discrepancy"  
> ここでは直近 30日を「テスト」として扱い、乖離の大きい特徴量を特定する

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def adversarial_validation(
    df: pd.DataFrame,
    feature_cols: list[str],
    date_col: str,
    recent_days: int = 90,
) -> pd.DataFrame:
    """
    直近 recent_days 日のデータを「テスト」として、
    それ以前との分布差を LightGBM で検出する。
    AUC > 0.6 の特徴量は分布シフトが疑われる。
    """
    max_date = df[date_col].max()
    threshold_date = max_date - pd.Timedelta(days=recent_days)

    df_adv = df[feature_cols].copy()
    label = (df[date_col] > threshold_date).astype(int)

    # 数値列のみ
    num_cols = df_adv.select_dtypes(include=[np.number]).columns.tolist()
    X_adv = df_adv[num_cols].fillna(-999)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X_adv))

    for tr_idx, va_idx in cv.split(X_adv, label):
        model = lgb.LGBMClassifier(
            n_estimators=100, learning_rate=0.1,
            max_depth=4, random_state=42, verbose=-1
        )
        model.fit(X_adv.iloc[tr_idx], label.iloc[tr_idx])
        oof_preds[va_idx] = model.predict_proba(X_adv.iloc[va_idx])[:, 1]

    auc = roc_auc_score(label, oof_preds)
    print(f"Adversarial AUC: {auc:.4f}  (0.5=no shift, >0.6=要注意)")

    # 特徴量重要度
    final_model = lgb.LGBMClassifier(
        n_estimators=100, learning_rate=0.1,
        max_depth=4, random_state=42, verbose=-1
    )
    final_model.fit(X_adv, label)
    imp = pd.DataFrame({
        "feature": num_cols,
        "importance": final_model.feature_importances_,
    }).sort_values("importance", ascending=False)

    return imp


if DATE_COL and len(df_train) > 1000:
    adv_imp = adversarial_validation(df_train, FEATURE_COLS, DATE_COL, recent_days=90)
    print("\n分布シフトが疑われる上位特徴量:")
    print(adv_imp.head(20).to_string(index=False))
else:
    adv_imp = pd.DataFrame()
    print("データが少ないため Adversarial Validation をスキップ")

## 6. Stage A: 3着以内分類モデル

> Equity HCT: "Component A — Predicts P(event=1)"  
> Child Mind: "Repeated Stratified KFold + 多シード" → ノイズに対してロバスト

### LGBM + XGB + CatBoost 3モデルアンサンブル

In [ ]:
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

N_SPLITS = 5
N_SEEDS = 3  # 多シード平均でノイズ安定化（Child Mind 金メダル戦略）

# 数値列のみ
X_num = X.select_dtypes(include=[np.number])
NUMERIC_FEATURE_COLS = X_num.columns.tolist()
print(f"数値特徴量数: {len(NUMERIC_FEATURE_COLS)}")

# LGBM params — DART boosting（AMEX 金メダル: "gold standard due to DART"）
LGBM_PARAMS_CLS = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "dart",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "num_leaves": 63,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "verbose": -1,
}

XGB_PARAMS_CLS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "verbosity": 0,
    "use_label_encoder": False,
}

CAT_PARAMS_CLS = {
    "iterations": 500,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": 0,
}

print("モデルパラメータ設定完了")

In [ ]:
def train_stage_a(
    X: pd.DataFrame,
    y: pd.Series,
    groups_date: pd.Series,
    n_splits: int = 5,
    n_seeds: int = 3,
) -> tuple[np.ndarray, list]:
    """
    Stage A（3着以内分類）を Purged GroupKFold × 多シードで学習。
    
    Returns:
        oof_preds: OOF 予測（Shape: len(X)）
        trained_models: 全フォールド × 全シードのモデルリスト
    """
    oof_preds = np.zeros(len(X))
    oof_counts = np.zeros(len(X))
    trained_models = []
    aucs = []

    X_arr = X.values.astype(np.float32)

    for seed in range(n_seeds):
        splitter = PurgedGroupTimeSeriesSplit(n_splits=n_splits, purge_gap_days=7)

        for fold_idx, (tr_idx, va_idx) in enumerate(splitter.split(X_arr, groups=groups_date)):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y.iloc[tr_idx].values, y.iloc[va_idx].values

            fold_preds = np.zeros(len(va_idx))
            n_models = 0

            # --- LGBM ---
            try:
                params = {**LGBM_PARAMS_CLS, "random_state": seed * 100 + fold_idx}
                lgbm_model = lgb.LGBMClassifier(**params)
                lgbm_model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_va, y_va)],
                    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
                )
                fold_preds += lgbm_model.predict_proba(X_va)[:, 1]
                n_models += 1
                trained_models.append(("lgbm", seed, fold_idx, lgbm_model))
            except Exception as e:
                print(f"  LGBM failed seed={seed} fold={fold_idx}: {e}")

            # --- XGBoost ---
            try:
                params = {**XGB_PARAMS_CLS, "random_state": seed * 100 + fold_idx}
                xgb_model = xgb.XGBClassifier(**params)
                xgb_model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_va, y_va)],
                    verbose=False,
                    early_stopping_rounds=50,
                )
                fold_preds += xgb_model.predict_proba(X_va)[:, 1]
                n_models += 1
                trained_models.append(("xgb", seed, fold_idx, xgb_model))
            except Exception as e:
                print(f"  XGB failed seed={seed} fold={fold_idx}: {e}")

            # --- CatBoost ---
            try:
                params = {**CAT_PARAMS_CLS, "random_seed": seed * 100 + fold_idx}
                cat_model = CatBoostClassifier(**params)
                cat_model.fit(
                    X_tr, y_tr,
                    eval_set=(X_va, y_va),
                    early_stopping_rounds=50,
                    verbose=False,
                )
                fold_preds += cat_model.predict_proba(X_va)[:, 1]
                n_models += 1
                trained_models.append(("cat", seed, fold_idx, cat_model))
            except Exception as e:
                print(f"  CatBoost failed seed={seed} fold={fold_idx}: {e}")

            if n_models > 0:
                fold_preds /= n_models
                oof_preds[va_idx] += fold_preds
                oof_counts[va_idx] += 1

                fold_auc = roc_auc_score(y_va, fold_preds)
                aucs.append(fold_auc)
                print(f"  Seed {seed} Fold {fold_idx+1}: AUC={fold_auc:.4f} (n_models={n_models})")

    # 平均化
    mask = oof_counts > 0
    oof_preds[mask] /= oof_counts[mask]

    overall_auc = roc_auc_score(y[mask], oof_preds[mask])
    print(f"\n=== Stage A OOF AUC: {overall_auc:.4f} (fold mean: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}) ===")

    return oof_preds, trained_models


print("train_stage_a() 定義完了")

In [ ]:
if DATE_COL and len(df_train) > 500:
    print("=== Stage A 学習開始 ===")
    oof_a, models_a = train_stage_a(
        X_num.loc[df_train.index],
        y_place,
        groups_date=df_train[DATE_COL],
        n_splits=5,
        n_seeds=N_SEEDS,
    )
    df_train["oof_stage_a"] = oof_a
else:
    print("データ不足のため Stage A をスキップ")
    oof_a = np.full(len(df_train), 0.3)
    df_train["oof_stage_a"] = oof_a
    models_a = []

## 7. Stage B: ランキングモデル（LambdaRank）

> Equity HCT: "Component B — Predicts ranking of event times"  
> OTTO Recommender: "LGBMRanker with LambdaRank loss"

着順ランキング学習。`group` パラメータ = 各レースの出走頭数。

In [ ]:
from scipy.stats import spearmanr


def train_stage_b(
    X: pd.DataFrame,
    y: pd.Series,
    race_ids: pd.Series,
    groups_date: pd.Series,
    n_splits: int = 5,
) -> np.ndarray:
    """
    Stage B（着順ランキング）を LambdaRank で学習。
    
    Returns:
        oof_scores: OOF ランキングスコア（低いほど上位予測）
    """
    oof_scores = np.zeros(len(X))
    oof_counts = np.zeros(len(X))
    spearman_list = []

    X_arr = X.values.astype(np.float32)

    splitter = PurgedGroupTimeSeriesSplit(n_splits=n_splits, purge_gap_days=7)

    for fold_idx, (tr_idx, va_idx) in enumerate(splitter.split(X_arr, groups=groups_date)):
        X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
        y_tr = y.iloc[tr_idx].values
        y_va = y.iloc[va_idx].values
        race_tr = race_ids.iloc[tr_idx]
        race_va = race_ids.iloc[va_idx]

        # グループ（各レースの頭数）
        group_tr = race_tr.value_counts()[race_tr.unique()].reindex(race_tr).values
        # シンプルに race_id でグループ数をカウント
        group_tr = race_tr.groupby(race_tr).transform("count").values
        # 重複を排除して正しいグループサイズ列を作る
        tr_df = pd.DataFrame({"race_id": race_tr, "y": y_tr})
        group_sizes_tr = tr_df.groupby("race_id", sort=False).size().values

        va_df = pd.DataFrame({"race_id": race_va, "y": y_va})

        # ランクターゲットを降順変換（着1位 = 最高スコア）
        y_tr_rank = -y_tr  # finish_pos が小さいほど良い → 符号反転

        ranker = lgb.LGBMRanker(
            objective="lambdarank",
            metric="ndcg",
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            num_leaves=63,
            min_child_samples=10,
            subsample=0.8,
            colsample_bytree=0.8,
            verbose=-1,
            random_state=42 + fold_idx,
        )
        try:
            ranker.fit(
                X_tr, y_tr_rank,
                group=group_sizes_tr,
            )
            fold_scores = ranker.predict(X_va)
            oof_scores[va_idx] += fold_scores
            oof_counts[va_idx] += 1

            # フォールドごとに Spearman 相関で評価
            corr, _ = spearmanr(y_va, -fold_scores)  # 着順と逆相関が正しい
            spearman_list.append(corr)
            print(f"  Fold {fold_idx+1}: Spearman={corr:.4f}")
        except Exception as e:
            print(f"  Fold {fold_idx+1}: LambdaRank failed: {e}")

    mask = oof_counts > 0
    oof_scores[mask] /= oof_counts[mask]

    if spearman_list:
        print(f"\n=== Stage B OOF Spearman: {np.mean(spearman_list):.4f} ± {np.std(spearman_list):.4f} ===")

    return oof_scores


print("train_stage_b() 定義完了")

In [ ]:
if DATE_COL and "race_id" in df_train.columns and len(df_train) > 500:
    print("=== Stage B 学習開始 ===")
    oof_b = train_stage_b(
        X_num.loc[df_train.index],
        y_rank,
        race_ids=df_train["race_id"],
        groups_date=df_train[DATE_COL],
        n_splits=5,
    )
    df_train["oof_stage_b"] = oof_b
else:
    print("データ不足のため Stage B をスキップ")
    oof_b = np.zeros(len(df_train))
    df_train["oof_stage_b"] = oof_b

## 8. 2段階スコア統合（Post-processing）

> Equity HCT 1位: `res = (1 - y_fun) * x_fun + y_fun`（Power Law Merge）  
> 競馬適用: Stage A（馬券圏内確率）× Stage B（ランキングスコア）を組み合わせる

### レース内整合後処理

> Optiver: "zero-sum post-processing — predictions should sum to zero"  
> IEEE Fraud: "UID Averaging — replace individual predictions with UID-level mean"

→ 競馬: 同レース内で予測スコアを相対スケールに正規化する

In [ ]:
def merge_stages(
    df: pd.DataFrame,
    oof_a: np.ndarray,
    oof_b: np.ndarray,
    alpha: float = 0.5,
) -> np.ndarray:
    """
    Stage A（3着内確率）と Stage B（ランキングスコア）を統合。
    
    Equity HCT 1位の Power Law Merge を参考に:
      final = alpha * norm(stage_a) + (1-alpha) * norm(stage_b)
    
    alpha=0.5 を基準とし、Optuna で最適化する。
    """
    from sklearn.preprocessing import MinMaxScaler

    # レース内での相対スケーリング（Optiver zero-sum 的な正規化）
    def race_normalize(scores: np.ndarray, race_ids: pd.Series) -> np.ndarray:
        normalized = np.zeros_like(scores)
        for race_id in race_ids.unique():
            mask = race_ids == race_id
            s = scores[mask]
            s_range = s.max() - s.min()
            if s_range > 0:
                normalized[mask] = (s - s.min()) / s_range
            else:
                normalized[mask] = 0.5
        return normalized

    if "race_id" in df.columns:
        norm_a = race_normalize(oof_a, df["race_id"])
        norm_b = race_normalize(oof_b, df["race_id"])
    else:
        scaler = MinMaxScaler()
        norm_a = scaler.fit_transform(oof_a.reshape(-1, 1)).ravel()
        norm_b = scaler.fit_transform(oof_b.reshape(-1, 1)).ravel()

    final_score = alpha * norm_a + (1 - alpha) * norm_b
    return final_score


final_score = merge_stages(df_train, oof_a, oof_b, alpha=0.5)
df_train["final_score"] = final_score

print("統合スコア shape:", final_score.shape)
print(f"スコア範囲: {final_score.min():.4f} 〜 {final_score.max():.4f}")

## 9. alpha 最適化（Nelder-Mead）

> Child Mind: "Optimized Thresholds using Nelder-Mead"  
> Equity HCT 5位: "Greedy Ensemble Selection"  

Stage A と Stage B の混合比 alpha をレース内的中率（3着以内予測精度）で最適化する。

In [ ]:
from scipy.optimize import minimize


def top3_precision_per_race(
    df: pd.DataFrame,
    score_col: str,
    true_col: str,
    top_n: int = 3,
) -> float:
    """
    各レースで上位 top_n 頭を予測し、実際に 3着以内だった割合を返す。
    """
    if "race_id" not in df.columns:
        return 0.0

    precisions = []
    for _, grp in df.groupby("race_id"):
        if len(grp) < top_n:
            continue
        top_predicted = grp.nlargest(top_n, score_col).index
        actual_top = grp[grp[true_col] <= 3].index
        hit = len(set(top_predicted) & set(actual_top))
        precisions.append(hit / top_n)

    return float(np.mean(precisions)) if precisions else 0.0


def objective_alpha(alpha_arr):
    alpha = float(np.clip(alpha_arr[0], 0, 1))
    score = merge_stages(df_train, oof_a, oof_b, alpha=alpha)
    df_tmp = df_train.copy()
    df_tmp["_score"] = score
    prec = top3_precision_per_race(df_tmp, "_score", TARGET_COL, top_n=3)
    return -prec  # 最小化


if TARGET_COL and "race_id" in df_train.columns:
    res = minimize(objective_alpha, x0=[0.5], method="Nelder-Mead",
                   options={"xatol": 0.01, "fatol": 0.001, "maxiter": 50})
    best_alpha = float(np.clip(res.x[0], 0, 1))
    best_prec = -res.fun
    print(f"最適 alpha: {best_alpha:.3f}")
    print(f"3着以内予測精度（上位3頭）: {best_prec:.4f}")

    # 最適 alpha で再計算
    final_score = merge_stages(df_train, oof_a, oof_b, alpha=best_alpha)
    df_train["final_score"] = final_score
else:
    best_alpha = 0.5
    print(f"最適化スキップ。alpha={best_alpha}")

## 10. 評価サマリー

In [ ]:
print("=" * 60)
print("Gold Medal Baseline 評価サマリー")
print("=" * 60)

# Stage A
if "oof_stage_a" in df_train.columns:
    mask_a = df_train["oof_stage_a"] > 0
    if mask_a.sum() > 0:
        auc_a = roc_auc_score(y_place[mask_a], df_train.loc[mask_a, "oof_stage_a"])
        print(f"\nStage A（3着以内分類）AUC:  {auc_a:.4f}")

# Stage B
if "oof_stage_b" in df_train.columns and TARGET_COL:
    mask_b = df_train["oof_stage_b"] != 0
    if mask_b.sum() > 0:
        corr, _ = spearmanr(y_rank[mask_b], -df_train.loc[mask_b, "oof_stage_b"])
        print(f"Stage B（ランキング）Spearman: {corr:.4f}")

# Final
if "final_score" in df_train.columns and "race_id" in df_train.columns:
    prec3 = top3_precision_per_race(df_train, "final_score", TARGET_COL, top_n=3)
    prec1 = top3_precision_per_race(df_train, "final_score", TARGET_COL, top_n=1)
    print(f"\n最終スコア（alpha={best_alpha:.2f}）:")
    print(f"  上位1頭的中率: {prec1:.4f}")
    print(f"  上位3頭的中率: {prec3:.4f}")

print("\n" + "=" * 60)
print("金メダル戦略 実装サマリー")
print("=" * 60)
print("""
✓ LGBM(DART) + XGBoost + CatBoost 3モデルアンサンブル
✓ 馬単位ラグ特徴（lag1〜3）+ ローリング統計（3/5走）
✓ 同日同開催コンテキスト集約（time_id 集約）
✓ ドメイン特徴（斤量負担率・速度指数トレンド）
✓ Purged GroupKFold（7日ギャップ付き時系列バリデーション）
✓ 多シード平均（N_SEEDS={} でノイズ安定化）
✓ Adversarial Validation（分布シフト検出）
✓ 2段階分解（Stage A: 分類 + Stage B: LambdaRank）
✓ Nelder-Mead alpha 最適化
✓ レース内スコア正規化（Optiver zero-sum 類似）
""".format(N_SEEDS))

## 11. 特徴量重要度（SHAP）

> AMEX 11位: "Null Importance for feature pruning"  
> Optiver 1位: "CatBoost feature importance to trim to 300 features"

In [ ]:
# LGBM 特徴量重要度（簡易版）
if models_a:
    lgbm_models = [m for name, seed, fold, m in models_a if name == "lgbm"]
    if lgbm_models:
        imp_all = np.zeros(len(NUMERIC_FEATURE_COLS))
        for m in lgbm_models:
            imp_all += m.feature_importances_
        imp_all /= len(lgbm_models)

        imp_df = pd.DataFrame({
            "feature": NUMERIC_FEATURE_COLS,
            "importance": imp_all,
        }).sort_values("importance", ascending=False)

        print("特徴量重要度 Top 30:")
        print(imp_df.head(30).to_string(index=False))

        # 低重要度特徴量の確認
        n_zero_imp = (imp_df["importance"] == 0).sum()
        print(f"\n重要度 0 の特徴量: {n_zero_imp}/{len(imp_df)} 件")
        print("→ 次回は上位特徴量のみで再学習することを推奨")
else:
    print("モデルが未学習のため特徴量重要度をスキップ")

## 12. 次のステップ

### 近期優先度の高い改善

| 戦略 | 内容 | 参考コンペ |
|---|---|---|
| **Optuna ハイパーパラメータ最適化** | LGBM/XGB/CatBoost のパラメータをベイズ最適化 | Mitsui 7位, ICR 6位 |
| **TabM / TabPFN 追加** | 小データサブセットでの精度向上 | Equity HCT 1〜6位全員, MCTS |
| **Online Learning** | 直近レース結果で週次再学習 | Optiver, Enefit |
| **Meta-features** | OOFスコアを特徴として2段スタック | AMEX, Home Credit |
| **Null Importance** | 特徴量プルーニング（重要度0を削除） | AMEX 11位 |
| **Greedy Ensemble Selection** | 多モデルから最良サブセット選択 | Equity HCT 5位 |

### 競馬固有の追加検討

1. **人気との交互作用**: 上位人気での着差特徴（過大/過少評価馬の検出）
2. **血統 × 馬場適性**: `track_bias_pedigree` を活用した芝/ダ・距離適性
3. **騎手・調教師統計のラグ**: 直近 N 週間の成績傾向  
4. **季節性特徴**: 月・クラス・開催場所の組み合わせ効果
